# Trabajo Final — Pipeline de pedidos de TiendaBSG
**Tópicos Avanzados para Data Science en Python**

**Alumno(a):** _Carla Bernachea Collazos_
**Repositorio público:** _https://github.com/carlabbc321/trabajo-final-spark.git_

---

Eres el *data engineer* de **TiendaBSG**, una tienda online con despachos en 6 ciudades del Perú. Cada día el sistema de ventas deja un archivo CSV con los pedidos del día en la carpeta `datos/raw/`. Tu trabajo es procesar los pedidos de **octubre 2026** de principio a fin:

| Parte | Qué harás | Capítulo | Puntos |
|---|---|---|---|
| 1. Revisar los archivos | Explorar `raw/`, separar archivos vacíos y contar registros | 1. Línea de comandos | 3.5 |
| 2. Analizar con Spark | Limpiar, filtrar, agrupar y contar con DataFrames o Spark SQL | 3. Spark | 12 |
| 3. Cerrar el proceso | Mover los CSV procesados y dejar el resultado final listo | 1. Línea de comandos | 2 |
| 4. Entregar | Subir este notebook y el resultado a tu repositorio público | 2. Git | 2.5 |
| | | **Total** | **20** |

**Reglas**
- Ejecuta las celdas **en orden**. Si algo se rompe, vuelve a ejecutar la **Preparación** (reinicia los datos) y continúa.
- En la Parte 2 usa solo **DataFrames** (API de PySpark o Spark SQL). **No uses RDDs.**
- Responde las preguntas como **comentarios** (`# ...`) en tu propia celda de código.
- Antes de descargar el notebook: **Entorno de ejecución → Reiniciar y ejecutar todo**, para que se vean todas las salidas en GitHub.

## Preparación (no se califica)
Ejecuta estas tres celdas. Instalan PySpark, crean la sesión de Spark y generan los archivos del mes:

```
/content/trabajo_final/
├── datos/
│   ├── raw/          ← 31 archivos pedidos_2026-10-DD.csv (uno por día)
│   ├── procesados/   ← vacía: aquí moverás los CSV ya procesados (Parte 3)
│   └── cuarentena/   ← vacía: aquí moverás los archivos con problemas (Parte 1)
└── resultados/       ← vacía: aquí guardarás el resumen final (Partes 2 y 3)
```

In [1]:
# Java + PySpark (tarda ~1 minuto la primera vez)
import shutil
if shutil.which("java") is None:
    !apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
!pip -q uninstall -y dataproc-spark-connect  # paquete de Colab que pide PySpark 4; no lo usamos
!pip -q install "pyspark==3.5.*"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("BSG-TrabajoFinal")
         .config("spark.ui.showConsoleProgress", "false")   # sin barras de progreso
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                     # oculta los logs de la JVM
print("Versión de Spark:", spark.version)

Versión de Spark: 3.5.9


In [3]:
# Genera los pedidos de octubre 2026 (volver a ejecutar = reiniciar los datos)
import csv, os, random
from datetime import date

TF = "/content/trabajo_final"
RAW = f"{TF}/datos/raw"
PROCESADOS = f"{TF}/datos/procesados"
CUARENTENA = f"{TF}/datos/cuarentena"
RESULTADOS = f"{TF}/resultados"

shutil.rmtree(TF, ignore_errors=True)
for carpeta in (RAW, PROCESADOS, CUARENTENA, RESULTADOS):
    os.makedirs(carpeta)

rng = random.Random(2026)
CIUDADES = {"Lima": 40, "Arequipa": 15, "Trujillo": 12, "Chiclayo": 10, "Cusco": 13, "Piura": 10}
CATALOGO = {  # producto: (categoria, precio de lista, peso en las ventas)
    "Laptop": ("Tecnologia", 3200.0, 8), "Celular": ("Tecnologia", 1500.0, 15),
    "Audifonos": ("Tecnologia", 180.0, 20), "Licuadora": ("Hogar", 250.0, 12),
    "Cafetera": ("Hogar", 320.0, 8), "Zapatillas": ("Moda", 280.0, 22),
    "Casaca": ("Moda", 220.0, 15)}
DEVOLUCION = {"Tecnologia": 4, "Hogar": 3, "Moda": 12}  # % de pedidos devueltos
PAGOS = {"Tarjeta": 40, "Yape": 30, "Efectivo": 15, "Transferencia": 15}
DIAS_VACIOS = {12, 25}  # el sistema falló y dejó estos archivos vacíos

id_pedido, dia_anterior = 0, []
for dia in range(1, 32):
    fecha = date(2026, 10, dia).isoformat()
    ruta = f"{RAW}/pedidos_{fecha}.csv"
    if dia in DIAS_VACIOS:
        open(ruta, "w").close()
        continue
    filas = []
    for _ in range(rng.randint(1500, 2500)):
        id_pedido += 1
        ciudad = rng.choices(list(CIUDADES), weights=CIUDADES.values())[0]
        producto = rng.choices(list(CATALOGO), weights=[v[2] for v in CATALOGO.values()])[0]
        categoria, precio_lista, _ = CATALOGO[producto]
        precio = round(precio_lista * rng.uniform(0.9, 1.1), 2)
        cantidad = rng.randint(1, 4)
        devolucion = DEVOLUCION[categoria]
        estado = rng.choices(["ENTREGADO", "CANCELADO", "DEVUELTO"],
                             weights=[90 - devolucion, 10, devolucion])[0]
        pago = rng.choices(list(PAGOS), weights=PAGOS.values())[0]
        # Errores típicos del sistema de origen
        r = rng.random()
        if r < 0.10:
            ciudad = ciudad.upper()
        elif r < 0.20:
            ciudad = f" {ciudad.lower()}  "
        if rng.random() < 0.05:
            estado = estado.lower()
        if rng.random() < 0.02:
            cantidad = ""
        if rng.random() < 0.01:
            precio = -precio
        filas.append([id_pedido, fecha, ciudad, categoria, producto, cantidad, precio, pago, estado])
    duplicados = rng.sample(dia_anterior, len(dia_anterior) // 50)  # pedidos reenviados
    dia_anterior = filas
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f, lineterminator="\n")
        w.writerow(["id_pedido", "fecha", "ciudad", "categoria", "producto",
                    "cantidad", "precio_unitario", "metodo_pago", "estado"])
        w.writerows(filas + duplicados)

%cd {TF}
!ls datos/raw | head -3
print("...")

/content/trabajo_final
pedidos_2026-10-01.csv
pedidos_2026-10-02.csv
pedidos_2026-10-03.csv
...


## Parte 1 — Revisar los archivos (Línea de comandos · 3.5 pts)
Antes de procesar nada, un *data engineer* revisa qué llegó. Usa **solo comandos de terminal** (`!comando` para una línea, o `%%bash` al inicio de la celda para varias). Ya estás ubicado en `/content/trabajo_final`.

| Comando | ¿Qué hace? |
|---|---|
| `ls -lh carpeta` | Lista archivos con su tamaño |
| `ls carpeta \| wc -l` | Cuenta cuántos archivos hay |
| `du -sh carpeta` | Tamaño total de una carpeta |
| `head -n 5 archivo` | Muestra las primeras líneas |
| `find carpeta -type f -empty` | Busca archivos vacíos |
| `mv origen destino` | Mueve (o renombra) archivos |
| `wc -l archivos` | Cuenta líneas |
| `grep -ci texto` | Cuenta líneas que contienen un texto (sin importar mayúsculas) |

### ✏️ Ejercicio 1.1 — Explorar la carpeta raw (1 pts)
1. Lista los archivos de `datos/raw` con su tamaño.
2. Cuenta cuántos archivos hay y el tamaño total de la carpeta.
3. Muestra las 5 primeras líneas del archivo del 1 de octubre.
4. Responde: ¿qué columnas tiene cada archivo? ¿Ves algún archivo raro en el listado?

In [4]:
%%bash
cd /content/trabajo_final/datos/raw

echo "1. Lista de archivos con sus tamaños:"
ls -lh

echo "2. Cantidad de archivos:"
find . -maxdepth 1 -type f -name "*.csv" | wc -l

echo "   Tamaño total de la carpeta:"
du -sh .

echo "3. Primeras 5 líneas del archivo del 1 de octubre:"
head -n 5 pedidos_2026-10-01.csv

echo "4. Columnas del archivo:"
head -n 1 pedidos_2026-10-01.csv

# ¿Ves algún archivo raro en el listado?
# Hay 2 archivos vacios

1. Lista de archivos con sus tamaños:
total 3.8M
-rw-r--r-- 1 root root 106K Nov 15 17:29 pedidos_2026-10-01.csv
-rw-r--r-- 1 root root 138K Nov 15 17:29 pedidos_2026-10-02.csv
-rw-r--r-- 1 root root 103K Nov 15 17:29 pedidos_2026-10-03.csv
-rw-r--r-- 1 root root 127K Nov 15 17:29 pedidos_2026-10-04.csv
-rw-r--r-- 1 root root 165K Nov 15 17:29 pedidos_2026-10-05.csv
-rw-r--r-- 1 root root 113K Nov 15 17:29 pedidos_2026-10-06.csv
-rw-r--r-- 1 root root 140K Nov 15 17:29 pedidos_2026-10-07.csv
-rw-r--r-- 1 root root 166K Nov 15 17:29 pedidos_2026-10-08.csv
-rw-r--r-- 1 root root 112K Nov 15 17:29 pedidos_2026-10-09.csv
-rw-r--r-- 1 root root 115K Nov 15 17:29 pedidos_2026-10-10.csv
-rw-r--r-- 1 root root 135K Nov 15 17:29 pedidos_2026-10-11.csv
-rw-r--r-- 1 root root    0 Nov 15 17:29 pedidos_2026-10-12.csv
-rw-r--r-- 1 root root 109K Nov 15 17:29 pedidos_2026-10-13.csv
-rw-r--r-- 1 root root 135K Nov 15 17:29 pedidos_2026-10-14.csv
-rw-r--r-- 1 root root 117K Nov 15 17:29 pedidos_2026-1

### ✏️ Ejercicio 1.2 — Separar los archivos vacíos (1.5 pts)
1. Encuentra los archivos vacíos de `datos/raw` con `find`.
2. Muévelos a `datos/cuarentena` (con `mv` o con `find ... -exec mv {} destino \;`).
3. Comprueba: lista `datos/cuarentena` y cuenta cuántos archivos quedan en `datos/raw`.
4. Responde: ¿por qué conviene apartarlos en lugar de borrarlos?

In [5]:
%%bash
cd /content/trabajo_final

echo "1. Archivos vacíos encontrados:"
find datos/raw -type f -empty

echo ""
echo "2. Moviendo archivos vacíos a cuarentena..."
find datos/raw -type f -empty -exec mv {} datos/cuarentena/ \;

echo ""
echo "3. Archivos en cuarentena:"
ls -lh datos/cuarentena

echo "Cantidad de archivos en cuarentena:"
find datos/cuarentena -maxdepth 1 -type f | wc -l

echo ""
echo "Cantidad de archivos restantes en raw:"
find datos/raw -maxdepth 1 -type f -name "*.csv" | wc -l

echo ""
# 4. ¿Por qué conviene apartarlos en lugar de borrarlos?
# Porque permite conservar evidencia de los archivos recibidos, investigar posibles errores y recuperarlos si fuera necesario.

1. Archivos vacíos encontrados:
datos/raw/pedidos_2026-10-12.csv
datos/raw/pedidos_2026-10-25.csv

2. Moviendo archivos vacíos a cuarentena...

3. Archivos en cuarentena:
total 0
-rw-r--r-- 1 root root 0 Nov 15 17:29 pedidos_2026-10-12.csv
-rw-r--r-- 1 root root 0 Nov 15 17:29 pedidos_2026-10-25.csv
Cantidad de archivos en cuarentena:
2

Cantidad de archivos restantes en raw:
29



### ✏️ Ejercicio 1.3 — Contar registros antes de Spark (1 pts)
1. Cuenta las líneas de todos los CSV de `datos/raw` juntos con `wc -l` (la última línea muestra el total).
2. Calcula cuántas **filas de datos** hay: total de líneas menos una cabecera por archivo. Anota el número: lo compararás en el Ejercicio 2.1.
3. Con `grep`, cuenta cuántas líneas contienen la palabra `cancelado` sin importar mayúsculas.

In [6]:
%%bash
cd /content/trabajo_final

echo "1. Total de líneas de todos los CSV:"
total_lineas=$(cat datos/raw/*.csv | wc -l)
echo "$total_lineas"

echo ""
echo "2. Cantidad de archivos CSV:"
archivos=$(find datos/raw -maxdepth 1 -type f -name "*.csv" | wc -l)
echo "$archivos"

filas_datos=$((total_lineas - archivos))
echo "Total de filas de datos (sin cabeceras): $filas_datos"

echo ""
echo "3. Líneas que contienen la palabra cancelado (sin distinguir mayúsculas):"
grep -ih "cancelado" datos/raw/*.csv | wc -l

1. Total de líneas de todos los CSV:
57332

2. Cantidad de archivos CSV:
29
Total de filas de datos (sin cabeceras): 57303

3. Líneas que contienen la palabra cancelado (sin distinguir mayúsculas):
5616


## Parte 2 — Analizar con Spark (DataFrames · 12 pts)
Cada ejercicio puedes resolverlo con la **API de DataFrames** o con **Spark SQL**; el Ejercicio 2.7 es obligatoriamente con SQL. **No uses RDDs.**

| API de DataFrames | Spark SQL | ¿Qué hace? |
|---|---|---|
| `df.select("a", "b")` | `SELECT a, b` | Elegir columnas |
| `df.filter(F.col("a") > 0)` | `WHERE a > 0` | Filtrar filas |
| `df.dropDuplicates(["id"])` | `SELECT DISTINCT ...` | Quitar duplicados |
| `df.dropna(subset=["a"])` | `WHERE a IS NOT NULL` | Quitar nulos |
| `df.withColumn("t", expr)` | `SELECT ..., expr AS t` | Crear o reemplazar una columna |
| `df.groupBy("a").agg(F.sum("b"))` | `GROUP BY a` | Agrupar y agregar |
| `df.groupBy("a").count()` | `COUNT(*)` | Contar por grupo |
| `df.orderBy(F.desc("b"))` | `ORDER BY b DESC` | Ordenar |
| `df.createOrReplaceTempView("v")` | `spark.sql("SELECT ... FROM v")` | Usar SQL sobre un DataFrame |

> 💡 Para que las sumas grandes no salgan en notación científica: `.cast("decimal(18,2)")` o `CAST(SUM(x) AS DECIMAL(18,2))`.

### ✏️ Ejercicio 2.1 — Leer los pedidos (1 pts)
1. Lee **todos** los CSV de `datos/raw` en un solo DataFrame llamado `pedidos` (pista: `spark.read.csv(f"{RAW}/*.csv", header=True, inferSchema=True)`).
2. Muestra el esquema con `printSchema()` y cuenta las filas.
3. Responde: ¿coincide con el número de filas de datos que calculaste en el Ejercicio 1.3?

In [7]:
# 1. Leer todos los CSV en un solo DataFrame
pedidos = spark.read.csv(f"{RAW}/*.csv",header=True,inferSchema=True)

# 2. Mostrar el esquema
print("Esquema del DataFrame:")
pedidos.printSchema()

# Contar las filas
total_pedidos = pedidos.count()
print("Total de filas leídas por Spark:", total_pedidos)

# 3. Comparar con el resultado del Ejercicio 1.3
# En el Ejercicio 1.3 obtuvimos 57.303 filas de datos.
if total_pedidos == 57303:
    print("La cantidad de filas coincide con el Ejercicio 1.3.")
else:
    print("La cantidad de filas NO coincide con el Ejercicio 1.3.")

Esquema del DataFrame:
root
 |-- id_pedido: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- metodo_pago: string (nullable = true)
 |-- estado: string (nullable = true)

Total de filas leídas por Spark: 57303
La cantidad de filas coincide con el Ejercicio 1.3.


### ✏️ Ejercicio 2.2 — Explorar la calidad de los datos (1.5 pts)
1. Muestra 5 filas solo de las columnas `fecha`, `ciudad`, `producto`, `cantidad` y `estado`.
2. Muestra los valores distintos de `ciudad` y cuéntalos. ¿Por qué hay más de 6?
3. Cuenta cuántas filas tienen `cantidad` nula y cuántas tienen `precio_unitario` menor o igual a 0.
4. Cuenta cuántos `id_pedido` están repetidos (filas totales menos `id_pedido` distintos).

In [8]:
from pyspark.sql import functions as F

# 1. Primeras 5 filas
print("1. Primeras 5 filas:")
pedidos.select("fecha", "ciudad", "producto", "cantidad", "estado").show(5, truncate=False)

# 2. Valores distintos de ciudad
print("\n2. Valores distintos de ciudad:")
ciudades = pedidos.select("ciudad").distinct()
ciudades.show(truncate=False)
print("Cantidad de ciudades distintas:", ciudades.count())

# Hay más de 6 porque algunos nombres de ciudades tienen diferencias de escritura o espacios adicionales.

# 3. Cantidad nula y precios inválidos
cantidad_nula = pedidos.filter(F.col("cantidad").isNull()).count()
precio_invalido = pedidos.filter(F.col("precio_unitario") <= 0).count()
print("\n3. Calidad de los datos:")
print("Filas con cantidad nula:", cantidad_nula)
print("Filas con precio menor o igual a 0:", precio_invalido)

# 4. id_pedido repetidos
filas_totales = pedidos.count()
ids_distintos = pedidos.select("id_pedido").distinct().count()
ids_repetidos = filas_totales - ids_distintos
print("\n4. id_pedido repetidos:")
print("Filas totales:", filas_totales)
print("id_pedido distintos:", ids_distintos)
print("id_pedido repetidos:", ids_repetidos)

1. Primeras 5 filas:
+----------+--------+----------+--------+---------+
|fecha     |ciudad  |producto  |cantidad|estado   |
+----------+--------+----------+--------+---------+
|2026-10-27|Lima    |Casaca    |2       |ENTREGADO|
|2026-10-27|Lima    |Zapatillas|1       |CANCELADO|
|2026-10-27|Trujillo|Cafetera  |4       |ENTREGADO|
|2026-10-27|CUSCO   |Audifonos |4       |ENTREGADO|
|2026-10-27|Arequipa|Audifonos |4       |ENTREGADO|
+----------+--------+----------+--------+---------+
only showing top 5 rows


2. Valores distintos de ciudad:
+-----------+
|ciudad     |
+-----------+
|Arequipa   |
|Lima       |
| arequipa  |
|Chiclayo   |
|Cusco      |
| lima      |
|AREQUIPA   |
|Piura      |
|Trujillo   |
| piura     |
|PIURA      |
|CHICLAYO   |
|CUSCO      |
| cusco     |
|TRUJILLO   |
| chiclayo  |
| trujillo  |
|LIMA       |
+-----------+

Cantidad de ciudades distintas: 18

3. Calidad de los datos:
Filas con cantidad nula: 1127
Filas con precio menor o igual a 0: 582

4. id_pedido

### ✏️ Ejercicio 2.3 — Limpiar los pedidos (2 pts)
Crea el DataFrame `df_limpio` a partir de `pedidos`:
1. Elimina los duplicados por `id_pedido`.
2. Elimina las filas con `cantidad` nula y con `precio_unitario <= 0`.
3. Normaliza `ciudad` (quita espacios y deja la primera letra en mayúscula: `F.initcap(F.trim(...))`).
4. Normaliza `estado` en mayúsculas y sin espacios (`F.upper(F.trim(...))`).
5. Agrega la columna `total = cantidad * precio_unitario` redondeada a 2 decimales.

Muestra cuántas filas había antes y después, y los valores distintos de `ciudad` (deben ser 6) y de `estado` (deben ser 3).

In [9]:
from pyspark.sql import functions as F

# 1 y 2. Eliminar duplicados, cantidades nulas y precios inválidos
df_limpio = pedidos.dropDuplicates(["id_pedido"])
df_limpio = df_limpio.filter(F.col("cantidad").isNotNull() & (F.col("precio_unitario") > 0))

# 3. Normalizar ciudad
df_limpio = df_limpio.withColumn("ciudad", F.initcap(F.trim(F.col("ciudad"))))

# 4. Normalizar estado
df_limpio = df_limpio.withColumn("estado", F.upper(F.trim(F.col("estado"))))

# 5. Calcular total
df_limpio = df_limpio.withColumn("total", F.round(F.col("cantidad") * F.col("precio_unitario"), 2))

# Mostrar resultados
print("1. Filas antes de la limpieza:", pedidos.count())
print("2. Filas después de la limpieza:", df_limpio.count())

print("\n3. Ciudades distintas:")
df_limpio.select("ciudad").distinct().show()

print("Cantidad de ciudades:", df_limpio.select("ciudad").distinct().count())

print("\n4. Estados distintos:")
df_limpio.select("estado").distinct().show()

print("Cantidad de estados:", df_limpio.select("estado").distinct().count())

1. Filas antes de la limpieza: 57303
2. Filas después de la limpieza: 54566

3. Ciudades distintas:
+--------+
|  ciudad|
+--------+
|Arequipa|
|    Lima|
|Chiclayo|
|   Cusco|
|   Piura|
|Trujillo|
+--------+

Cantidad de ciudades: 6

4. Estados distintos:
+---------+
|   estado|
+---------+
|CANCELADO|
|ENTREGADO|
| DEVUELTO|
+---------+

Cantidad de estados: 3


### ✏️ Ejercicio 2.4 — Filtrar pedidos (1.5 pts)
1. Crea el DataFrame `entregados` solo con los pedidos en estado `ENTREGADO` y cuéntalos.
2. ¿Cuántos pedidos entregados en **Lima**, pagados con **Yape**, tienen un `total` mayor a **1000**?
3. Muestra los 5 pedidos entregados de mayor `total` (columnas `id_pedido`, `ciudad`, `producto`, `cantidad`, `total`).

In [10]:
# 1. Pedidos entregados
print("1. Pedidos entregados:")
entregados = df_limpio.filter(F.col("estado") == "ENTREGADO")
entregados.show(5, truncate=False)
print("Total de pedidos entregados:", entregados.count())

# 2. Pedidos de Lima pagados con Yape mayores a S/ 1000
print("\n2. Pedidos de Lima pagados con Yape mayores a S/ 1000:")
lima_yape = df_limpio.filter(
    (F.col("ciudad") == "Lima") & (F.col("metodo_pago") == "Yape") & (F.col("total") > 1000)
)
lima_yape.show(5, truncate=False)
print("Total de pedidos encontrados:", lima_yape.count())

# 3. Top 5 pedidos por total
print("\n3. Top 5 pedidos por total:")
top5 = df_limpio.orderBy(F.col("total").desc())
top5.show(5, truncate=False)

1. Pedidos entregados:
+---------+----------+--------+----------+----------+--------+---------------+-------------+---------+-------+
|id_pedido|fecha     |ciudad  |categoria |producto  |cantidad|precio_unitario|metodo_pago  |estado   |total  |
+---------+----------+--------+----------+----------+--------+---------------+-------------+---------+-------+
|3        |2026-10-01|Chiclayo|Moda      |Zapatillas|3       |263.77         |Transferencia|ENTREGADO|791.31 |
|6        |2026-10-01|Arequipa|Moda      |Zapatillas|4       |283.58         |Efectivo     |ENTREGADO|1134.32|
|9        |2026-10-01|Cusco   |Hogar     |Licuadora |3       |239.72         |Tarjeta      |ENTREGADO|719.16 |
|12       |2026-10-01|Lima    |Hogar     |Licuadora |4       |229.83         |Transferencia|ENTREGADO|919.32 |
|13       |2026-10-01|Trujillo|Tecnologia|Celular   |1       |1386.25        |Transferencia|ENTREGADO|1386.25|
+---------+----------+--------+----------+----------+--------+---------------+-----------

### ✏️ Ejercicio 2.5 — Conteos (1.5 pts)
1. Cuenta los pedidos por `estado`, de mayor a menor.
2. Cuenta los pedidos por `metodo_pago`, de mayor a menor.
3. Responde: ¿qué porcentaje de los pedidos se cancela? ¿Cuál es el método de pago más usado?

In [11]:
# 1. Pedidos por estado, ordenados de mayor a menor
print("1. Pedidos por estado:")
df_limpio.groupBy("estado").count().orderBy(F.col("count").desc()).show()

# 2. Pedidos por método de pago, ordenados de mayor a menor
print("\n2. Pedidos por método de pago:")
conteo_pago = df_limpio.groupBy("metodo_pago").count().orderBy(F.col("count").desc())
conteo_pago.show()

# 3. Porcentaje de cancelaciones y método de pago más usado
total_pedidos = df_limpio.count()
pedidos_cancelados = df_limpio.filter(F.col("estado") == "CANCELADO").count()
porcentaje_cancelaciones = (pedidos_cancelados / total_pedidos) * 100

metodo_mas_usado = conteo_pago.first()

print("\n3. Respuestas:")
print("Porcentaje de pedidos cancelados:", round(porcentaje_cancelaciones, 2), "%")
print("Método de pago más usado:", metodo_mas_usado["metodo_pago"])

1. Pedidos por estado:
+---------+-----+
|   estado|count|
+---------+-----+
|ENTREGADO|45665|
|CANCELADO| 5339|
| DEVUELTO| 3562|
+---------+-----+


2. Pedidos por método de pago:
+-------------+-----+
|  metodo_pago|count|
+-------------+-----+
|      Tarjeta|21958|
|         Yape|16287|
|     Efectivo| 8211|
|Transferencia| 8110|
+-------------+-----+


3. Respuestas:
Porcentaje de pedidos cancelados: 9.78 %
Método de pago más usado: Tarjeta


### ✏️ Ejercicio 2.6 — Ventas por ciudad (2 pts)
Con los pedidos **entregados**, crea el DataFrame `ventas_por_ciudad` con una fila por ciudad y estas columnas:
- `pedidos`: número de pedidos
- `unidades`: suma de `cantidad`
- `ventas`: suma de `total` (en decimal, sin notación científica)
- `ticket_promedio`: promedio de `total` redondeado a 2 decimales

Ordénalo de mayor a menor `ventas` y muéstralo. Responde: ¿qué ciudad vende más y cuál menos?

In [12]:
# 1. Filtrar únicamente los pedidos entregados
pedidos_entregados = df_limpio.filter(F.col("estado") == "ENTREGADO")

# 2. Calcular pedidos, unidades, ventas y ticket promedio por ciudad
ventas_por_ciudad = pedidos_entregados.groupBy("ciudad").agg(
    F.count("id_pedido").alias("pedidos"),
    F.sum("cantidad").alias("unidades"),
    F.sum(F.col("total").cast("decimal(18,2)")).cast("decimal(18,2)").alias("ventas"),
    F.round(F.avg("total"), 2).cast("decimal(18,2)").alias("ticket_promedio")
)

# 3. Ordenar de mayor a menor según las ventas
ventas_por_ciudad = ventas_por_ciudad.orderBy(F.col("ventas").desc())

# 4. Mostrar resultados
print("1. Ventas por ciudad:")
ventas_por_ciudad.show(truncate=False)

# 5. Identificar las ciudades con mayores y menores ventas
print("2. Ciudad con mayores ventas:", ventas_por_ciudad.first()["ciudad"])
print("3. Ciudad con menores ventas:", ventas_por_ciudad.orderBy(F.col("ventas").asc()).first()["ciudad"])


1. Ventas por ciudad:
+--------+-------+--------+-----------+---------------+
|ciudad  |pedidos|unidades|ventas     |ticket_promedio|
+--------+-------+--------+-----------+---------------+
|Lima    |18341  |45793   |31123284.87|1696.92        |
|Arequipa|6760   |16824   |11215911.73|1659.16        |
|Cusco   |5961   |14913   |9804943.29 |1644.85        |
|Trujillo|5483   |13648   |8951107.45 |1632.52        |
|Piura   |4554   |11480   |7852829.32 |1724.38        |
|Chiclayo|4566   |11282   |7766903.83 |1701.03        |
+--------+-------+--------+-----------+---------------+

2. Ciudad con mayores ventas: Lima
3. Ciudad con menores ventas: Chiclayo


## ✏️ Ejercicio 2.7 — Las mismas preguntas con Spark SQL (1.5 pts)
**Este ejercicio es obligatorio con SQL.**
1. Registra `df_limpio` como la vista temporal `pedidos_limpios`.
2. Con SQL, obtén por `categoria` el número de pedidos **entregados** y la suma de sus ventas, de mayor a menor.
3. Con SQL, obtén por `categoria` el número de pedidos **devueltos**, de mayor a menor. Responde: ¿qué categoría tiene más devoluciones?

In [13]:
# 1. Registrar el DataFrame como vista temporal
df_limpio.createOrReplaceTempView("pedidos_limpios")

# 2. Ventas por categoría de pedidos entregados
ventas_categoria = spark.sql("""
    SELECT categoria,
           COUNT(id_pedido) AS pedidos,
           CAST(SUM(total) AS DECIMAL(18,2)) AS ventas
    FROM pedidos_limpios
    WHERE estado = 'ENTREGADO'
    GROUP BY categoria
    ORDER BY ventas DESC
""")

print("1. Ventas por categoría:")
ventas_categoria.show(truncate=False)

# 3. Cantidad de pedidos devueltos por categoría
devoluciones_categoria = spark.sql("""
    SELECT categoria,
           COUNT(id_pedido) AS devoluciones
    FROM pedidos_limpios
    WHERE estado = 'DEVUELTO'
    GROUP BY categoria
    ORDER BY devoluciones DESC
""")

print("\n2. Devoluciones por categoría:")
devoluciones_categoria.show(truncate=False)

# 4. Identificar la categoría con más devoluciones
categoria_mas_devoluciones = devoluciones_categoria.first()["categoria"]

print("\n3. Categoría con más devoluciones:", categoria_mas_devoluciones)

1. Ventas por categoría:
+----------+-------+-----------+
|categoria |pedidos|ventas     |
+----------+-------+-----------+
|Tecnologia|20216  |59937691.17|
|Moda      |15713  |10007489.55|
|Hogar     |9736   |6769799.77 |
+----------+-------+-----------+


2. Devoluciones por categoría:
+----------+------------+
|categoria |devoluciones|
+----------+------------+
|Moda      |2340        |
|Tecnologia|873         |
|Hogar     |349         |
+----------+------------+


3. Categoría con más devoluciones: Moda


### ✏️ Ejercicio 2.8 — Guardar el resultado (1 pts)
1. Guarda `ventas_por_ciudad` como CSV con cabecera en `resultados/ventas_por_ciudad_spark`, en **un solo archivo** (pista: `coalesce(1)` y `mode("overwrite")`).
2. Lista la carpeta que creó Spark. Responde: ¿por qué Spark crea una **carpeta** y no un archivo?

In [14]:
# 1. Guardar las ventas por ciudad como CSV en un solo archivo
ventas_por_ciudad.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{RESULTADOS}/ventas_por_ciudad_spark")

# 2. Listar el contenido de la carpeta creada por Spark
print("1. Contenido de la carpeta:")
!ls -lh {RESULTADOS}/ventas_por_ciudad_spark

# 3. ¿Por qué Spark crea una carpeta y no un archivo?
# Porque Spark trabaja de forma distribuida y organiza los resultados de escritura en una carpeta.
# Aunque usamos coalesce(1) para generar un solo archivo CSV, Spark mantiene esa estructura de carpeta,
# donde guarda el archivo generado y otros archivos de control."

1. Contenido de la carpeta:
total 4.0K
-rw-r--r-- 1 root root 274 Nov 15 17:30 part-00000-45f23d6b-a82d-495b-81d9-542f4daedd4f-c000.csv
-rw-r--r-- 1 root root   0 Nov 15 17:30 _SUCCESS


## Parte 3 — Cerrar el proceso (Línea de comandos · 2 pts)
Spark ya terminó. Ahora deja las carpetas ordenadas para el próximo mes: lo que está en `raw/` ya se procesó y el resultado debe tener un nombre claro.

### ✏️ Ejercicio 3.1 — Mover los CSV procesados (1 pts)
1. Mueve todos los CSV de `datos/raw` a `datos/procesados` con `mv`.
2. Comprueba que `datos/raw` quedó vacía y cuenta cuántos archivos hay en `datos/procesados`.
3. Responde: ¿por qué es importante sacar los archivos de `raw/` después de procesarlos?

In [15]:
%%bash
cd /content/trabajo_final

# 1. Mover todos los CSV de raw a procesados
if compgen -G "datos/raw/*.csv" > /dev/null; then
    mv datos/raw/*.csv datos/procesados/
fi

# 2. Comprobar que raw quedó vacía y contar los archivos en procesados
echo "1. Archivos restantes en raw:"
ls -A datos/raw

echo "Cantidad de archivos en procesados:"
find datos/procesados -maxdepth 1 -type f -name "*.csv" | wc -l

# 3. ¿Por qué es importante sacar los archivos de raw/ después de procesarlos?
# Porque evita que los archivos ya procesados vuelvan a procesarse por error
# y permite diferenciar los archivos pendientes de los que ya fueron procesados.

1. Archivos restantes en raw:
Cantidad de archivos en procesados:
29


### ✏️ Ejercicio 3.2 — Dejar el resultado con un nombre claro (1 pts)
1. Mueve el archivo `part-*.csv` que creó Spark a `resultados/ventas_por_ciudad.csv` (con `mv` también se renombra).
2. Borra la carpeta `resultados/ventas_por_ciudad_spark`.
3. Muestra el contenido de `resultados/ventas_por_ciudad.csv` y la estructura final con `find datos resultados -type f | sort`.

In [16]:
%%bash
cd /content/trabajo_final

# 1. Mover y renombrar el archivo CSV generado por Spark
mv resultados/ventas_por_ciudad_spark/part-*.csv resultados/ventas_por_ciudad.csv

# 2. Borrar la carpeta creada por Spark
rm -r resultados/ventas_por_ciudad_spark

# 3. Mostrar el contenido del archivo CSV
echo "3. Contenido de ventas_por_ciudad.csv:"
cat resultados/ventas_por_ciudad.csv

# Mostrar la estructura final de archivos
echo ""
echo "Estructura final:"
find datos resultados -type f | sort

3. Contenido de ventas_por_ciudad.csv:
ciudad,pedidos,unidades,ventas,ticket_promedio
Lima,18341,45793,31123284.87,1696.92
Arequipa,6760,16824,11215911.73,1659.16
Cusco,5961,14913,9804943.29,1644.85
Trujillo,5483,13648,8951107.45,1632.52
Piura,4554,11480,7852829.32,1724.38
Chiclayo,4566,11282,7766903.83,1701.03

Estructura final:
datos/cuarentena/pedidos_2026-10-12.csv
datos/cuarentena/pedidos_2026-10-25.csv
datos/procesados/pedidos_2026-10-01.csv
datos/procesados/pedidos_2026-10-02.csv
datos/procesados/pedidos_2026-10-03.csv
datos/procesados/pedidos_2026-10-04.csv
datos/procesados/pedidos_2026-10-05.csv
datos/procesados/pedidos_2026-10-06.csv
datos/procesados/pedidos_2026-10-07.csv
datos/procesados/pedidos_2026-10-08.csv
datos/procesados/pedidos_2026-10-09.csv
datos/procesados/pedidos_2026-10-10.csv
datos/procesados/pedidos_2026-10-11.csv
datos/procesados/pedidos_2026-10-13.csv
datos/procesados/pedidos_2026-10-14.csv
datos/procesados/pedidos_2026-10-15.csv
datos/procesados/pedidos_202

## Parte 4 — Entregar con Git (2.5 pts)
Esta parte se hace **en la terminal de tu computadora** (Git Bash en Windows, Terminal en Mac/Linux), siguiendo los pasos del documento del trabajo final.

1. Descarga el resultado ejecutando la celda de abajo (se guarda en tu carpeta de Descargas).
2. Escribe tu nombre y el link de tu repositorio al inicio de este notebook.
3. **Entorno de ejecución → Reiniciar y ejecutar todo**, y luego **Archivo → Descargar → Descargar .ipynb**.
4. Sube los dos archivos a tu repositorio público con `git add`, `git commit` y `git push`.

In [17]:
from google.colab import files
files.download(f"{RESULTADOS}/ventas_por_ciudad.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>